# 💻 Unidad 4: Material Complementario - Práctica
## Módulo 02 - Monitoreo de Modelos en Producción
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos de la Práctica

En esta práctica vas a:

1. ✅ Simular datos de entrenamiento y producción
2. ✅ Detectar Data Drift usando KS test
3. ✅ Calcular Population Stability Index (PSI)
4. ✅ Visualizar cambios en distribuciones
5. ✅ Detectar Concept Drift monitoreando accuracy
6. ✅ Crear un dashboard de monitoreo

---

### 📋 Ejercicios

1. **Ejercicio 1**: Detectar Data Drift con KS Test
2. **Ejercicio 2**: Calcular PSI para estabilidad poblacional
3. **Ejercicio 3**: Visualizar distribuciones (con/sin drift)
4. **Ejercicio 4**: Detectar Concept Drift con métricas de modelo
5. **Ejercicio 5**: Dashboard de resumen de monitoreo

---

### ⏱️ Duración Estimada: 60 minutos

## 🛠️ Setup: Instalación de Librerías

Instalamos las librerías necesarias para monitoreo de modelos:

In [0]:
# Instalar librerías para monitoreo
%pip install scipy scikit-learn

print("✅ Librerías instaladas")

In [0]:
# Imports
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

---

## 📊 Generación de Datos de Entrenamiento

**Objetivo**: Crear un dataset simulado que represente los datos de entrenamiento original

**Características**:
* Feature X: Distribución normal (media=50, desv=10)
* Target y: Clasificación binaria basada en threshold
* 1,000 muestras

In [0]:
# Generar datos de entrenamiento
np.random.seed(42)
n_train = 1000
X_train = np.random.normal(loc=50, scale=10, size=n_train)
y_train = (X_train + np.random.randn(n_train) * 5 > 55).astype(int)

print("✅ Datos de entrenamiento generados")
print(f"  Media de X: {X_train.mean():.2f}")
print(f"  Desviación estándar: {X_train.std():.2f}")
print(f"  Tasa positiva: {y_train.mean():.2%}")

---

## 🔍 Ejercicio 1: Detectar Data Drift con KS Test

**Objetivo**: Usar el test de Kolmogorov-Smirnov para detectar cambios en la distribución

**Método**:
* Comparar distribución de entrenamiento vs producción
* KS test: mide máxima diferencia entre CDFs
* P-value < 0.05 indica drift significativo

**Escenarios**:
1. Producción sin drift (misma distribución)
2. Producción con drift (distribución cambiada)

In [0]:
print("🔍 Ejercicio 1: Detectar Data Drift\n" + "="*60)

# Simular datos de producción con y sin drift
n_prod = 500
X_prod_no_drift = np.random.normal(loc=50, scale=10, size=n_prod)
X_prod_with_drift = np.random.normal(loc=60, scale=12, size=n_prod)  # ⚠️ Drift

# KS Test para detectar drift
ks_stat_no_drift, p_value_no_drift = ks_2samp(X_train, X_prod_no_drift)
ks_stat_with_drift, p_value_with_drift = ks_2samp(X_train, X_prod_with_drift)

print("\n📊 Escenario 1: Sin Drift")
print(f"  KS Statistic: {ks_stat_no_drift:.4f}")
print(f"  P-value: {p_value_no_drift:.4f}")
print(f"  Resultado: {'✅ Sin drift' if p_value_no_drift > 0.05 else '⚠️ Drift detectado'}")

print("\n📊 Escenario 2: Con Drift")
print(f"  KS Statistic: {ks_stat_with_drift:.4f}")
print(f"  P-value: {p_value_with_drift:.4f}")
print(f"  Resultado: {'✅ Sin drift' if p_value_with_drift > 0.05 else '⚠️ Drift detectado'}")

print("\n💡 Interpretación:")
print("  • P-value > 0.05: No hay evidencia de drift")
print("  • P-value < 0.05: Drift detectado (distribuciones diferentes)")

---

## 📊 Ejercicio 2: Calcular Population Stability Index (PSI)

**Objetivo**: Medir la estabilidad de la población usando PSI

**Método**:
* PSI compara distribuciones por bins/percentiles
* Útil para monitoreo continuo

**Interpretación**:
* PSI < 0.1: Sin drift ✅
* 0.1 ≤ PSI < 0.2: Drift moderado ⚠️
* PSI ≥ 0.2: Drift significativo ❌

In [0]:
print("📊 Ejercicio 2: Calcular PSI\n" + "="*60)

def calculate_psi(expected, actual, bins=10):
    """Calcula Population Stability Index"""
    expected_counts, bin_edges = np.histogram(expected, bins=bins)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    # Evitar división por cero
    expected_pct = (expected_counts + 1) / (len(expected) + bins)
    actual_pct = (actual_counts + 1) / (len(actual) + bins)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

psi_no_drift = calculate_psi(X_train, X_prod_no_drift)
psi_with_drift = calculate_psi(X_train, X_prod_with_drift)

print("\n📊 Escenario 1: Sin Drift")
print(f"  PSI: {psi_no_drift:.4f}")
if psi_no_drift < 0.1:
    print("  Interpretación: ✅ Sin drift")
elif psi_no_drift < 0.2:
    print("  Interpretación: ⚠️ Drift moderado")
else:
    print("  Interpretación: ❌ Drift significativo")

print("\n📊 Escenario 2: Con Drift")
print(f"  PSI: {psi_with_drift:.4f}")
if psi_with_drift < 0.1:
    print("  Interpretación: ✅ Sin drift")
elif psi_with_drift < 0.2:
    print("  Interpretación: ⚠️ Drift moderado")
else:
    print("  Interpretación: ❌ Drift significativo")

print("\n💡 PSI permite monitoreo continuo de la estabilidad poblacional")

---

## 📊 Ejercicio 3: Visualizar Distribuciones

**Objetivo**: Comparar visualmente distribuciones de training vs producción

**Visualizaciones**:
1. Histogramas superpuestos
2. Comparación lado a lado

**Beneficio**: Detectar visualmente patrones de drift que los tests numéricos confirman

In [0]:
print("📊 Ejercicio 3: Visualizar distribuciones\n" + "="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escenario 1: Sin drift
axes[0].hist(X_train, bins=30, alpha=0.5, label='Train', density=True, color='blue')
axes[0].hist(X_prod_no_drift, bins=30, alpha=0.5, label='Prod (sin drift)', density=True, color='green')
axes[0].set_title('📊 Sin Data Drift', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Valor de Feature')
axes[0].set_ylabel('Densidad')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].text(0.5, 0.95, '✅ Distribuciones similares', 
             transform=axes[0].transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

# Escenario 2: Con drift
axes[1].hist(X_train, bins=30, alpha=0.5, label='Train', density=True, color='blue')
axes[1].hist(X_prod_with_drift, bins=30, alpha=0.5, label='Prod (con drift)', density=True, color='red')
axes[1].set_title('⚠️ Con Data Drift', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Valor de Feature')
axes[1].set_ylabel('Densidad')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].text(0.5, 0.95, '⚠️ Distribuciones diferentes', 
             transform=axes[1].transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
display(plt.gcf())
plt.close()

print("\n✅ Visualización completada")
print("💡 Las diferencias visuales ayudan a identificar el tipo de drift")

---

## 🔄 Ejercicio 4: Detectar Concept Drift

**Objetivo**: Detectar cambios en la relación X → Y (concepto)

**Diferencia clave**:
* **Data Drift**: Distribución de X cambia
* **Concept Drift**: Relación X → Y cambia (mismo X, diferente Y)

**Método**:
1. Entrenar modelo con datos originales
2. Evaluar en producción
3. Monitorear degradación de accuracy/métricas

In [0]:
print("🔄 Ejercicio 4: Detectar Concept Drift\n" + "="*60)

# Entrenar modelo simple
model = LogisticRegression()
model.fit(X_train.reshape(-1, 1), y_train)

print("✅ Modelo de clasificación entrenado")
print(f"  Tipo: {type(model).__name__}")
print(f"  Threshold original: ~55 (implícito en generación de y_train)")

In [0]:
# Escenario 1: Sin concept drift (misma relación X->Y)
y_prod_no_concept_drift = (X_prod_no_drift + np.random.randn(n_prod) * 5 > 55).astype(int)
pred_no_concept_drift = model.predict(X_prod_no_drift.reshape(-1, 1))
acc_no_concept_drift = accuracy_score(y_prod_no_concept_drift, pred_no_concept_drift)

print("\n📊 Escenario 1: Sin Concept Drift")
print(f"  Accuracy: {acc_no_concept_drift:.4f} ✅")
print(f"  El modelo mantiene su desempeño")

# Escenario 2: Con concept drift (threshold cambió de 55 a 45)
y_prod_with_concept_drift = (X_prod_no_drift + np.random.randn(n_prod) * 5 > 45).astype(int)
pred_with_concept_drift = model.predict(X_prod_no_drift.reshape(-1, 1))
acc_with_concept_drift = accuracy_score(y_prod_with_concept_drift, pred_with_concept_drift)

print("\n📊 Escenario 2: Con Concept Drift")
print(f"  Accuracy: {acc_with_concept_drift:.4f} ❌")
print(f"  Degradación: {(acc_no_concept_drift - acc_with_concept_drift)*100:.1f}%")

if acc_with_concept_drift < acc_no_concept_drift - 0.05:
    print("\n⚠️ ALERTA: Concept drift detectado")
    print("💡 Acción recomendada: Reentrenar modelo con datos recientes")
else:
    print("\n✅ No se detectó concept drift significativo")

---

## 📊 Ejercicio 5: Dashboard de Monitoreo

**Objetivo**: Crear un resumen ejecutivo de todas las métricas de monitoreo

**Componentes**:
1. Data Drift (KS test, PSI)
2. Concept Drift (Accuracy)
3. Estado general del modelo

**Uso en producción**:
* Este dashboard se actualizaría automáticamente
* Alertas se enviarían cuando se detecte drift
* Ayuda a decidir cuándo reentrenar

In [0]:
print("📊 Ejercicio 5: Dashboard de Monitoreo\n" + "="*60)

monitoring_summary = pd.DataFrame({
    'Métrica': [
        'KS Statistic (Data Drift)',
        'PSI (Data Drift)',
        'Accuracy (Concept Drift)'
    ],
    'Sin Drift': [
        f"{ks_stat_no_drift:.4f}",
        f"{psi_no_drift:.4f}",
        f"{acc_no_concept_drift:.4f}"
    ],
    'Con Drift': [
        f"{ks_stat_with_drift:.4f}",
        f"{psi_with_drift:.4f}",
        f"{acc_with_concept_drift:.4f}"
    ],
    'Estado': [
        '⚠️ Drift' if p_value_with_drift < 0.05 else '✅ OK',
        '❌ Drift Significativo' if psi_with_drift > 0.2 else '✅ OK',
        '⚠️ Degradación' if acc_with_concept_drift < 0.75 else '✅ OK'
    ]
})

print("\n📋 Dashboard de Monitoreo de Modelo:\n")
print(monitoring_summary.to_string(index=False))

print("\n" + "="*60)
print("✅ Ejercicios de monitoreo completados")
print("\n💡 Puntos clave:")
print("  • Data Drift: Cambios en distribución de features (KS, PSI)")
print("  • Concept Drift: Cambios en relación X→Y (accuracy, F1)")
print("  • En producción: Monitoreo automático + alertas")
print("  • Acción: Reentrenar cuando se detecte drift significativo")

---

## ✅ Resumen de la Práctica

### 🎯 Conceptos Aprendidos

1. **Data Drift**
   * KS Test para detectar cambios en distribución
   * PSI para monitoreo continuo de estabilidad
   * Umbrales: p-value < 0.05, PSI > 0.2

2. **Concept Drift**
   * Monitoreo de métricas de modelo (accuracy, F1, etc.)
   * Detectar cambios en relación X → Y
   * Degradación > 5% requiere acción

3. **Monitoreo en Producción**
   * Dashboard automático
   * Alertas proactivas
   * Decisión de reentrenamiento

### 🚀 Próximos Pasos

* Implementar monitoreo en Databricks Jobs
* Configurar alertas automáticas
* Integrar con MLflow para tracking
* Automatizar reentrenamiento cuando se detecte drift

---

**Universidad del Aconcagua 🇦🇷**